In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

#mental health data import (https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health?resource=download)
#download dataset, upload to google drive, then replace file path below
df = pd.read_csv('/content/drive/MyDrive/Combined Data.csv')
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety


In [3]:
#basic preprocessing

import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["clean_text"] = df["statement"].apply(clean_text)

In [4]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['status'])

print(encoder.classes_)

['Anxiety' 'Bipolar' 'Depression' 'Normal' 'Personality disorder' 'Stress'
 'Suicidal']


In [5]:
#train test split (0.7/0.3)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['clean_text'], df['label'], test_size=0.3, random_state=42, stratify=df['label'])

In [6]:
#tuning

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('logreg', LogisticRegression(
        class_weight = 'balanced',
        max_iter = 1000,
        random_state = 42
    ))
])

param_grid = {
    'tfidf__max_features': [5000, 10000, 20000, 40000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [1, 2, 3, 5],
    'tfidf__max_df': [0.75, 0.85, 0.9, 0.95],
    'tfidf__sublinear_tf': [True, False],
    'tfidf__use_idf': [True, False],
    'tfidf__norm': ['l2', None],
    'tfidf__binary': [True, False],
    'tfidf__stop_words': [None, 'english'],
    'logreg__C': [0.1, 1, 10],
    'logreg__penalty': ['l1', 'l2'],
    'logreg__solver': ['liblinear']
}

randomized_search = RandomizedSearchCV(pipeline, param_distributions=param_grid, n_iter=40, scoring='f1_weighted', cv=5, random_state=42, n_jobs=-1)
randomized_search.fit(X_train, y_train)

print("Best params:")
print(randomized_search.best_params_)

print("\nBest crossval score:")
print(randomized_search.best_score_)

Best params:
{'tfidf__use_idf': False, 'tfidf__sublinear_tf': False, 'tfidf__stop_words': None, 'tfidf__norm': None, 'tfidf__ngram_range': (1, 2), 'tfidf__min_df': 2, 'tfidf__max_features': 40000, 'tfidf__max_df': 0.85, 'tfidf__binary': True, 'logreg__solver': 'liblinear', 'logreg__penalty': 'l2', 'logreg__C': 0.1}

Best crossval score:
0.7609441226941657


In [8]:
#predictions

best_model = randomized_search.best_estimator_

y_pred = best_model.predict(X_test)

y_prob = best_model.predict_proba(X_test)

In [9]:
#eval

from sklearn.metrics import classification_report

print(classification_report(y_test,y_pred,target_names=encoder.classes_))

from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))

                      precision    recall  f1-score   support

             Anxiety       0.81      0.81      0.81      1167
             Bipolar       0.83      0.77      0.80       863
          Depression       0.75      0.68      0.71      4621
              Normal       0.90      0.93      0.92      4905
Personality disorder       0.62      0.76      0.68       360
              Stress       0.57      0.62      0.59       801
            Suicidal       0.66      0.70      0.68      3196

            accuracy                           0.77     15913
           macro avg       0.73      0.75      0.74     15913
        weighted avg       0.77      0.77      0.77     15913

[[ 949   14   50   58   29   62    5]
 [  25  663   66   27   43   29   10]
 [  76   66 3131  169   37   94 1048]
 [  39   14   69 4563   14  143   63]
 [   9    8   38    9  272   21    3]
 [  63   25   86   70   40  493   24]
 [   7   10  743  166    4   27 2239]]


In [11]:
import numpy as np

print(encoder.classes_)
print(np.unique(y_train))
print(np.unique(y_pred))

labels, counts = np.unique(y_pred, return_counts=True)

print(labels)
print(counts)

print(X_train.shape)

print(pd.Series(y_pred).value_counts())

['Anxiety' 'Bipolar' 'Depression' 'Normal' 'Personality disorder' 'Stress'
 'Suicidal']
[0 1 2 3 4 5 6]
[0 1 2 3 4 5 6]
[0 1 2 3 4 5 6]
[1168  800 4183 5062  439  869 3392]
(37130,)
3    5062
2    4183
6    3392
0    1168
5     869
1     800
4     439
Name: count, dtype: int64


In [13]:
#indicators

best_model = grid_search.best_estimator_

feature_names = best_model.named_steps["tfidf"].get_feature_names_out()

coef = best_model.named_steps["logreg"].coef_

for i, c in enumerate(encoder.classes_):
    top = coef[i].argsort()[-20:]

    print(c)
    print(feature_names[top])

Anxiety
['infection' 'googling' 'hiv' 'pressure' 'worries' 'scared' 'waiting'
 'worrying' 'heart' 'health anxiety' 'health' 'symptoms' 'cancer' 'worry'
 'restlessness' 'nervous' 'worried' 'anxious' 'anxiety' 'restless']
Bipolar
['pdoc' 'depressed' 'episodes' 'mood' 'abilify' 'depression that'
 'illness' 'seroquel' 'bp' 'hypomania' 'stable' 'latuda' 'lamictal'
 'lithium' 'episode' 'hypomanic' 'mania' 'meds' 'manic' 'bipolar']
Depression
['na' 'parent' 'thing' 'antidepressants' 'le' 'http co' 'depressive'
 'symptom' 'pression' 'did not' 'antidepressant' 'not' 'ha' 'doe' 'am'
 'cannot' 'depressed' 'do not' 'wa' 'depression']
Normal
['dream' 'you should' 'me too' 'though' 'homeless' 'bored' 'twitter'
 'holiday' 'ptsd' 'yes' 'na' 'quot' 'haha' 'had been' 'ðÿ' 'eid' 'met'
 'ha' 'url' 'wa']
Personality disorder
['avoidance' 'ghosting' 'is your' 'my avpd' 'with avpd' 'don' 'shame'
 'social' 'avoiding' 'this disorder' 'interaction' 'do you' 'people' 'nan'
 'hypochondria' 'poll' 'view poll' 'avo

In [ ]:
#try crossval and hyperparameter tuning